# 12S pipeline comparisons
- DADA2 & VSEARCH _de nova_ with RDP and BLAST taxonomy assignment
- BLASTed against both Meta Fish Lib and NCBI, while assignTaxonomy() against Meta Fish Lib
- In total, six comparisons (2 denoising methods x 3 taxonomy assignment methods)
- BLAST results condensed by taxonomizr
- DADA2 pipeline based on NEOF version, although updated. 
- Non-target taxa explored by taxonomy assignment to MIDORI2
- Compares ASVs, sequences through the pipeline, and taxa (species, genus and family)

## Set-up

Install packages, load required functions and get data.

In [0]:
source("Scripts/00_setup.R", echo = FALSE) #packages and functions

In [0]:
getwd()
list.files()
file.exists("Scripts/00_setup.R")

In [0]:
%sh
# cutadapt
pip install -q cutadapt
cutadapt --version

# BLAST+ installed
apt-get -q update
apt-get -q install -y ncbi-blast+

# location for cutadapt
which cutadapt

In [0]:
#temp solution
cutadapt_loc <-"/local_disk0/.ephemeral_nfs/envs/pythonEnv-a0bdf837-a7bf-4431-a07d-0ff6dbf96e73/bin/cutadapt"

In [0]:
test_data_loc <- paste("Data/Raw/RingTrial_Sean")
results_loc = paste("Results")
print(head(list.files(test_data_loc)))

In [0]:
manifest <- make_manifest(test_data_loc)
write.csv(manifest, "Data/Temp/manifest.csv")
head(manifest)

## Removing Ns

In [0]:
source("Scripts/01_remove_Ns.R")

## Identify and remove primers

In [0]:
#MiFish-U, Miya et al. (2015)

FWD <- "ACTGGGATTAGATACCCC"
REV <- "TAGAACAGGCTCCTCTAG"

In [0]:
# these would be options in actual pipeline

minimum <- 60 #Minimum read length cutoff. Recommend >0
copies <- 2 #Number of copies of a primer to be removed as sometimes dulication can occur. Recommended minimum is 2

In [0]:
source("Scripts/02_primer_removal.R")

## Generate quality plots

In [0]:
source("Scripts/03_raw_quality_plots.R")

Grey-scale heatmap shows the frequency of each quality score along the read lengths - looking for over 30 ish. Green line is median quality score. Orange line are quartiles. Red line at the bottom represent the proportion of reads of that particular length. 

Our reads on average are approx 100bp long after primers removed in the first two samples.

In [0]:
print(plotQualityProfile(fnFs.cut[1:2]))

In [0]:
print(plotQualityProfile(fnRs.cut[1:2]))

## Cleaning the data (filterAndTrim)

Parameters for filterAndTrim include:
- **maxN** - after truncation, sequences with more then X Ns will be disgarded
- **truncQ** - Truncate reads at the first instance of a quality score less then or equal to X
- **rm.phix** - Discard reads that match against the phiX genome
- **maxEE** - After truncation, reads with higher than X expected errors will be discarded
- **minLen** - Remove reads with length less than 60 (note these should gave already been removed by cutadapt)
- **multithread** - input files are filtered in parallel (logical)
- **truncLen** - controls where each read is cut (truncated) based on its length

In [0]:
truncLen=c(80,95) # where the quality dropped in previous plots
maxEE <-  c(2,2) #maxEE value 
truncQ <- 2 #truncQ value
minLen <- 50 #Impose a minimum length cutoff

#other options in NEOF pipeline are subset (only run a portion of the data) and marker (for better labelling when running multiple markers)

In [0]:
source("Scripts/04_filterAndTrim.R")

In [0]:
plotQualityProfile(fnFs.filtN[5:6])

In [0]:
plotQualityProfile(fnRs.filtN[1:2])

## Generate error model

In [0]:
source("Scripts/05_generate_error_model.R")

In the plots below:
- Error rates for each possible transition (e.g. A->C, A->G) are shown
- Red line = expected based on quality score
- Black line = estimate 
- Black dots = observed 

We want black lines and black dots to match. We also want to see a rough negative correlation. 

If it looks weird, can increase the number of bases the function is using (default is 100 million).

In [0]:
plotErrors(errF, nominalQ = TRUE)


In [0]:
plotErrors(errR, nominalQ = TRUE)

Common to see 'hook' between 30 - 40 with NovaSeq and MiniSeq data when visualising in DADA2. Couple of forums have discussion on this topic. Not to be of too much concern unless seeing any weird results.

## Deplication, merging and chimera removal

### DADA2

In [0]:
source("Scripts/06_derep_DADA2.R")

### Sequence tracking

In [0]:
source("Scripts/07_sequence_tracking.R")

### Assign taxonomy

Used standard Meta-Fish-Lib for now. Jono to make custom database from fish work.

In [0]:
source("Scripts/08_assign_taxonomy_RDP.R")

Now, let's try BLAST, using Meta Fish Lib and NCBI.

In [0]:
%sh
# timeout error currently, maybe use split and cat (but this is a small dataset so shouldn't be an issue)
bash Scripts/08b_assign_taxonomy_BLAST.sh

In [0]:
%sh
# timeout error currently, maybe use split and cat (but this is a small dataset so shouldn't be an issue)

# define paths
db_src="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/12S_fish_db"
db_tmp="/tmp/blastdb"
db_name="12S_fish_db"
query="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/06_ASV_seqs_DADA2.fasta"
output_MFL="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/08b_ASVs_blast_Metafishlib.txt"
output_NCBI="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/08b_ASVs_blast_NCBI.txt"

# copy BLAST database to local disk (doesn't work on DASH without doing this)
mkdir -p $db_tmp

cp ${db_src}/${db_name}.* $db_tmp/

# sanity check
ls -lh $db_tmp

# blast against meta fish lib
blastn -query $query -db refseq_rna -remote -task blastn -evalue 1 -outfmt 6 -perc_identity 97 -out $output_NCBI 

In [0]:
taxa_BLAST_MFL <- read.table("Data/Processed/08b_ASVs_blast_Metafishlib.txt", sep = "\t", header = FALSE)

colnames(taxa_BLAST_MFL) <- c(
  "qseqid", "sseqid", "pident", "length",
  "mismatch", "gapopen", "qstart", "qend",
  "sstart", "send", "evalue", "bitscore"
)

str(taxa_BLAST_MFL)

In [0]:
taxa_BLAST_NCBI <- read.table("Data/Processed/08b_ASVs_blast_NCBI.txt", sep = "\t", header = FALSE)

colnames(taxa_BLAST_NCBI) <- c(
  "qseqid", "sseqid", "pident", "length",
  "mismatch", "gapopen", "qstart", "qend",
  "sstart", "send", "evalue", "bitscore"
)

str(taxa_BLAST_NCBI)

Condense taxonomy (lowest common ancestor) for BLAST outputs

In [0]:
#also a timeout issue (both remote and direct download)

%sh
wget https://ftp.ncbi.nlm.nih.gov/pub/taxonomy/accession2taxid/nucl_gb.accession2taxid.gz


In [0]:
# get Database
options(timeout = 6000)  # increase to ~100 minutes
taxonomizr::prepareDatabase('accessionTaxa.sql')

In [0]:
source("Scripts/09_condensing_taxonomy.R")

### Create tidy datasets

Create both long and phyloseq formats.

In [0]:
taxa.print <- taxa # Removing sequence rownames for display only
rownames(taxa.print) <- NULL
head(taxa.print)

### Explore comparisons


## Jono pipeline

In [0]:
%skip
%python
%run Scripts/00_setup.py #packages and functions

In [0]:
%skip
%sh
pip install /Workspace/Shared/monitoring/EA_diatom_dada2_pipeline_DASH/packages/dokdo

In [0]:
%skip
%python
test_data_loc = "Data/Raw/RingTrial_Sean/"
results_loc = "Results"
os.listdir(test_data_loc)[:5]

In [0]:
%skip
%python
file_paths = get_files(results_loc)
samples = sort_paths(file_paths, results_loc)
export(samples, results_loc)

In [0]:
%skip
%sh
#define the analysis location (not retained from other cell)
results_loc = "/Results/"

qiime tools import \
    --type 'SampleData[PairedEndSequencesWithQuality]' \
    --input-path $results_loc/pe-33-manifest \
    --output-path /tmp/paired-end-demux.qza \
    --input-format PairedEndFastqManifestPhred33V2 && rm $results_loc/*fastq.gz

qiime demux summarize \
    --i-data /tmp/paired-end-demux.qza \
    --o-visualization /tmp/paired-end-demux.qzv

mv /tmp/paired-end-demux.qza $results_loc/paired-end-demux.qza
mv /tmp/paired-end-demux.qzv $results_loc/paired-end-demux.qzv